# Per-relation Router Complementarity Analysis

This notebook analyzes a diagnostic router, not the current PathBSR model.

The router is selected on validation relations: for each relation, choose the model with the best validation MRR among
`AnyBURL`, `ConvE`, `HoGRN`, `PathBSR`, `TransE`, and `TuckER`. Then evaluate that fixed per-relation choice on the test split.

This supports a future-work / complementarity discussion:

- Does per-relation specialization beat PathBSR alone?
- Does it beat the best single model selected on validation?
- How far is it from the per-query oracle upper bound?

Required command:

```bash
PYTHONPATH=src .venv/bin/python scripts/router_analysis.py
```

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / "results"

MODEL_ORDER = ["AnyBURL", "ConvE", "HoGRN", "PathBSR", "TransE"]
DATASET_ORDER = ["FB15K-237-10", "FB15K-237-20", "FB15K-237-50", "NELL23K", "WD-singer", "WN18RR"]

ROUTER = RESULTS / "router"

In [ ]:
summary_path = ROUTER / "per_relation_router_test_summary.csv"
figure_path = ROUTER / "per_relation_router_test_summary.png"
if not summary_path.exists() or not figure_path.exists():
    display(Markdown("""
**Missing generated artifact.** Run:

```bash
PYTHONPATH=src .venv/bin/python scripts/router_analysis.py
```
"""))
else:
    summary = pd.read_csv(summary_path)
    summary["dataset"] = pd.Categorical(summary["dataset"], DATASET_ORDER, ordered=True)
    display(summary.sort_values("dataset").round(4))
    display(Image(filename=str(figure_path)))

In [ ]:
# Relation assignment counts selected on validation.
assignment_path = ROUTER / "per_relation_router_assignment_counts.csv"
if assignment_path.exists():
    assignment = pd.read_csv(assignment_path)
    display(
        assignment.pivot_table(
            index="dataset",
            columns="selected_model",
            values="relations",
            aggfunc="sum",
            fill_value=0,
        )
    )

In [ ]:
# Relation-level selections with validation MRR.
relation_path = ROUTER / "per_relation_router_selected_on_valid.csv"
if relation_path.exists():
    relation = pd.read_csv(relation_path)
    display(relation.sort_values(["dataset", "selected_valid_mrr"], ascending=[True, False]).head(30))

## Interpretation guide

Use this as an analysis of model complementarity, not as a replacement for the current PathBSR result.
The valid-selected per-relation router is more realistic than an oracle, but it can still overfit relations with very few validation queries.
The oracle is only an upper bound because it chooses the best model per test query.

Good paper framing:

> Per-relation routing reveals that PathBSR and embedding/neural baselines make complementary errors. The remaining gap to the oracle suggests that future work could learn a stronger query-aware router rather than replacing PathBSR's interpretable reasoning core.